# Micro-NAS v3: Comparação das 5 SSt — CIFAR-10

Dataset: **CIFAR-10** (5.000 treino / 2.000 validação, low-fidelity proxy)  
Orçamento: **100 avaliações reais** por SSt  
Métricas: **Acurácia**, **F1-score macro**, **FLOPs estimados**  

SSt comparadas:
1. Random Search (RS)
2. Evolutionary Algorithm (EA)
3. Reinforcement Learning (RL) — REINFORCE com entropia
4. Bayesian Optimization (BO) — GP + EI
5. Gradient-based (GO) — DARTS-inspired


## 1. Definição do SSp e da VSt

### Search Space (SSp) — expandido

SSp com 7 dimensões, totalizando `3×3×2×3×3×3×2 = 972` combinações:

| Hiperparâmetro | Opções |
|---|---|
| `num_layers` | 1, 2, 3 |
| `filters` | 32, 64, 128 |
| `kernel_size` | 3, 5 |
| `activation` | relu, elu, tanh |
| `dropout` | 0.1, 0.3, 0.5 |
| `dense_units` | 64, 128, 256 |
| `optimizer` | adam, sgd_momentum |

### VSt — Partial Training no CIFAR-10

Cada candidato é treinado por **10 épocas** com EarlyStopping(patience=4).  
A função retorna acurácia, F1-score macro e FLOPs estimados.  
O sinal de busca das SSt é a acurácia; F1 e FLOPs são métricas de análise final.


In [ ]:
import os
import logging
# Força o uso do alocador assíncrono para mitigar vazamentos de VRAM
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import random
import time
import gc
import csv
import tensorflow as tf
from tensorflow.keras import layers, models, Input, optimizers, backend as K
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from sklearn.metrics import f1_score

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
tf.get_logger().setLevel(logging.ERROR)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
print("Carregando CIFAR-10...")
(x_tr_full, y_tr_full), (x_te_full, y_te_full) = tf.keras.datasets.cifar10.load_data()

# Mantém em uint8 na RAM
x_train = x_tr_full[:5000]
y_train = y_tr_full[:5000].flatten()
x_val   = x_te_full[:2000]
y_val   = y_te_full[:2000].flatten()

del x_tr_full, y_tr_full, x_te_full, y_te_full
gc.collect()

# Criação do pipeline tf.data otimizado
def preprocess_data(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

BATCH_SIZE = 64
train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_ds = train_ds.map(preprocess_data, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((x_val, y_val))
val_ds = val_ds.map(preprocess_data, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print(f"Shape de treino: {x_train.shape} | Shape de validação: {x_val.shape}")

Carregando CIFAR-10...
170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
Shape de treino: (5000, 32, 32, 3) | Shape de validação: (2000, 32, 32, 3)


In [ ]:
search_space = {
    'num_layers':  [1, 2, 3],
    'filters':     [32, 64, 128],
    'kernel_size': [3, 5],
    'activation':  ['relu', 'elu', 'tanh'],
    'dropout':     [0.1, 0.3, 0.5],
    'dense_units': [64, 128, 256],
    'optimizer':   ['adam', 'sgd_momentum'],
}

def gerar_aleatorio():
    return {k: random.choice(v) for k, v in search_space.items()}

def dict_to_vec(arq):
    return [search_space[k].index(arq[k]) for k in search_space]

def estimar_flops(params):
    H, W, C = 32, 32, 3
    macs = 0
    for i in range(params['num_layers']):
        cout = min(params['filters'] * (2 ** i), 256)
        macs += H * W * C * cout * params['kernel_size'] ** 2
        C = cout
        H = max(H // 2, 1)
        W = max(W // 2, 1)
    macs += H * W * C * params['dense_units']
    macs += params['dense_units'] * 10
    return int(2 * macs)

In [ ]:
def avaliar_arquitetura(params):
    # 1. Limpeza no início da execução
    K.clear_session()
    gc.collect()

    inp = Input(shape=(32, 32, 3))
    x = inp
    for i in range(params['num_layers']):
        f = min(params['filters'] * (2 ** i), 256)
        x = layers.Conv2D(f, params['kernel_size'],
                          activation=params['activation'], padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D((2, 2), padding='same')(x)

    x = layers.Flatten()(x)
    x = layers.Dropout(params['dropout'])(x)
    x = layers.Dense(params['dense_units'], activation=params['activation'])(x)
    out = layers.Dense(10, activation='softmax')(x)
    model = models.Model(inp, out)

    opt = (optimizers.SGD(learning_rate=0.01, momentum=0.9)
           if params['optimizer'] == 'sgd_momentum' else 'adam')

    model.compile(optimizer=opt, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit(train_ds, epochs=10, validation_data=val_ds, verbose=0,
              callbacks=[tf.keras.callbacks.EarlyStopping(
                  monitor='val_accuracy', patience=4, restore_best_weights=True)])

    # 2. Contorno do memory leak: Chamada direta ao invés de model.predict()
    y_pred_list = []
    for x_batch, _ in val_ds:
        # Passar os dados diretamente pela rede (training=False desativa o Dropout/BN dinâmico)
        preds = model(x_batch, training=False)
        y_pred_list.extend(np.argmax(preds.numpy(), axis=1))

    y_pred = np.array(y_pred_list)

    acc   = float(np.mean(y_pred == y_val))
    f1    = float(f1_score(y_val, y_pred, average='macro', zero_division=0))
    flops = estimar_flops(params)

    # 3. Limpeza agressiva no final da execução
    del model
    del y_pred_list
    del preds

    K.clear_session()

    # Força a coleta de referências cíclicas profundas
    for _ in range(3):
        gc.collect()

    return acc, f1, flops

def avaliar_acc(params):
    acc, _, _ = avaliar_arquitetura(params)
    return acc

## 2. Implementação das SSt

### 2.1 Random Search (RS) — baseline

### 2.2 Evolutionary Algorithm (EA) — mutação adaptativa + população top-5
Taxa de mutação decresce de 0.4→0.1 para explorar no início e explotar no fim.
Seleção por torneio entre os 3 melhores para manter diversidade.

### 2.3 Reinforcement Learning (RL) — REINFORCE + regularização de entropia
Bônus de entropia `β·H(π)` previne colapso prematuro da política (β decai linearmente 0.1→0).

### 2.4 Bayesian Optimization (BO) — GP + EI com warm-up de 10 avaliações
Pool de 500 candidatos por iteração. Kernel Matérn-5/2.

### 2.5 Gradient-based (GO) — DARTS com diferenças centrais + cosine annealing
Gradiente por diferenças centrais `(f(x+δ) - f(x-δ)) / 2δ`.
Learning rate com cosine annealing para estabilidade na convergência.


In [ ]:
max_avaliacoes = 100
resultados = {}

def random_search(max_evals):
    random.seed(SEED); np.random.seed(SEED)
    melhor_acc, melhor_arq = 0, None
    for i in range(max_evals):
        arq = gerar_aleatorio()
        acc = avaliar_acc(arq)
        if acc > melhor_acc:
            melhor_acc, melhor_arq = acc, arq
        print(f"  RS [{i+1:3d}/{max_evals}] acc={acc:.4f} best={melhor_acc:.4f}")
    return melhor_arq, melhor_acc

def evolutionary_search(max_evals):
    random.seed(SEED); np.random.seed(SEED)
    pop_size = 5
    populacao = []
    for i in range(pop_size):
        arq = gerar_aleatorio()
        acc = avaliar_acc(arq)
        populacao.append({'arq': arq, 'acc': acc})
        print(f"  EA [{i+1:3d}/{max_evals}] acc={acc:.4f} (init)")
    evals = pop_size

    while evals < max_evals:
        taxa_mutacao = 0.4 - 0.3 * (evals / max_evals)
        populacao.sort(key=lambda x: x['acc'], reverse=True)
        pai = random.choice(populacao[:3])
        mae = random.choice(populacao[:3])
        filho_arq = {k: pai['arq'][k] if random.random() > 0.5 else mae['arq'][k]
                     for k in search_space}
        for gene in list(search_space.keys()):
            if random.random() < taxa_mutacao:
                filho_arq[gene] = random.choice(search_space[gene])
        acc = avaliar_acc(filho_arq)
        populacao.append({'arq': filho_arq, 'acc': acc})
        populacao.sort(key=lambda x: x['acc'], reverse=True)
        populacao = populacao[:pop_size + 1]
        evals += 1
        print(f"  EA [{evals:3d}/{max_evals}] acc={acc:.4f} best={populacao[0]['acc']:.4f}")

    return populacao[0]['arq'], populacao[0]['acc']

def rl_search(max_evals):
    """
    REINFORCE v2 — melhorias sobre v1:
      1. lr 0.4 -> 0.15  (evita explosão dos logits)
      2. baseline EMA (alpha=0.1) em vez de média acumulada
      3. normalização de R por std rolling (janela=20)
      4. beta entropia decai até 0.02, não até 0 (mantém exploração mínima)
      5. clip dos logits em [-5, 5] (impede colapso numérico)
    """
    random.seed(SEED); np.random.seed(SEED)
    lr         = 0.15
    beta_start = 0.1
    beta_end   = 0.02
    ema_alpha  = 0.1
    clip_val   = 5.0

    theta    = {k: np.zeros(len(v)) for k, v in search_space.items()}
    ema_base = None
    rewards  = []
    melhor_acc, melhor_arq = 0, None

    def softmax(x):
        e = np.exp(x - np.max(x)); return e / e.sum()

    for t in range(max_evals):
        beta = beta_start + (beta_end - beta_start) * (t / max(max_evals - 1, 1))

        arq, indices = {}, {}
        for k, opts in search_space.items():
            pi  = softmax(theta[k])
            idx = np.random.choice(len(opts), p=pi)
            arq[k] = opts[idx]; indices[k] = idx

        acc = avaliar_acc(arq)
        rewards.append(acc)

        if ema_base is None:
            ema_base = acc
        else:
            ema_base = (1 - ema_alpha) * ema_base + ema_alpha * acc

        R_raw  = acc - ema_base
        window = rewards[-20:]
        std    = float(np.std(window)) if len(window) > 1 else 1.0
        R      = R_raw / (std + 1e-8)

        for k, opts in search_space.items():
            pi         = softmax(theta[k])
            grad_logpi = -pi.copy(); grad_logpi[indices[k]] += 1.0
            grad_h     = np.log(pi + 1e-8) + 1.0
            theta[k]  += lr * (R * grad_logpi + beta * grad_h)
            theta[k]   = np.clip(theta[k], -clip_val, clip_val)

        if acc > melhor_acc:
            melhor_acc, melhor_arq = acc, arq

        pi_layers  = softmax(theta['num_layers'])
        pi_filters = softmax(theta['filters'])
        print(f"  RL [{t+1:3d}/{max_evals}] acc={acc:.4f} best={melhor_acc:.4f} "
              f"base={ema_base:.4f} beta={beta:.3f} | "
              f"layers pi={np.round(pi_layers,2)} filters pi={np.round(pi_filters,2)}")

    return melhor_arq, melhor_acc

def bayesian_search(max_evals):
    random.seed(SEED); np.random.seed(SEED)
    xi = 0.01
    gp = GaussianProcessRegressor(
        kernel=Matern(nu=2.5), alpha=1e-6,
        normalize_y=True, n_restarts_optimizer=5)
    hist_X, hist_y = [], []
    melhor_acc, melhor_arq = 0, None

    def ei(X_cand):
        mu, sigma = gp.predict(X_cand, return_std=True)
        sigma  = np.maximum(sigma, 1e-9)
        f_star = np.max(hist_y)
        Z = (mu - f_star - xi) / sigma
        return (mu - f_star - xi) * norm.cdf(Z) + sigma * norm.pdf(Z)

    for i in range(10):
        arq = gerar_aleatorio()
        acc = avaliar_acc(arq)
        hist_X.append(dict_to_vec(arq)); hist_y.append(acc)
        if acc > melhor_acc: melhor_acc, melhor_arq = acc, arq
        print(f"  BO [{i+1:3d}/{max_evals}] acc={acc:.4f} (warmup)")

    for i in range(10, max_evals):
        gp.fit(hist_X, hist_y)
        cands   = [gerar_aleatorio() for _ in range(500)]
        ei_vals = ei([dict_to_vec(c) for c in cands])
        best_c  = cands[np.argmax(ei_vals)]
        acc     = avaliar_acc(best_c)
        hist_X.append(dict_to_vec(best_c)); hist_y.append(acc)
        if acc > melhor_acc: melhor_acc, melhor_arq = acc, best_c
        print(f"  BO [{i+1:3d}/{max_evals}] acc={acc:.4f} best={melhor_acc:.4f}")

    return melhor_arq, melhor_acc

def darts_search(max_evals):
    random.seed(SEED); np.random.seed(SEED)
    lr_max, lr_min, delta = 0.5, 0.05, 0.5
    alpha  = {k: np.zeros(len(v), dtype=float) for k, v in search_space.items()}
    n_keys = len(search_space)
    evals  = 0
    melhor_acc, melhor_arq = 0, None

    def softmax(x):
        e = np.exp(x - np.max(x)); return e / e.sum()

    def arq_de_alpha(a):
        return {k: search_space[k][int(np.argmax(softmax(a[k])))] for k in search_space}

    acc_base = avaliar_acc(arq_de_alpha(alpha)); evals += 1
    melhor_acc, melhor_arq = acc_base, arq_de_alpha(alpha)
    print(f"  GO [{evals:3d}/{max_evals}] acc={acc_base:.4f} (init)")

    step = 0
    max_steps = max_evals // (2 * n_keys + 1)
    while evals + 2 * n_keys + 1 <= max_evals:
        lr = lr_min + 0.5 * (lr_max - lr_min) * (1 + np.cos(np.pi * step / max(max_steps, 1)))
        grad = {k: np.zeros(len(v)) for k, v in search_space.items()}

        for k in search_space:
            if evals + 2 > max_evals: break
            i_p = int(np.argmax(softmax(alpha[k])))
            alpha[k][i_p] += delta
            a_plus = arq_de_alpha(alpha); acc_plus = avaliar_acc(a_plus); evals += 1
            if acc_plus > melhor_acc: melhor_acc, melhor_arq = acc_plus, a_plus
            print(f"  GO [{evals:3d}/{max_evals}] acc={acc_plus:.4f} best={melhor_acc:.4f}")
            alpha[k][i_p] -= 2 * delta
            a_minus = arq_de_alpha(alpha); acc_minus = avaliar_acc(a_minus); evals += 1
            if acc_minus > melhor_acc: melhor_acc, melhor_arq = acc_minus, a_minus
            print(f"  GO [{evals:3d}/{max_evals}] acc={acc_minus:.4f} best={melhor_acc:.4f}")
            alpha[k][i_p] += delta
            grad[k][i_p] = (acc_plus - acc_minus) / (2 * delta)

        for k in search_space:
            alpha[k] += lr * grad[k]

        if evals < max_evals:
            arq = arq_de_alpha(alpha)
            acc_base = avaliar_acc(arq); evals += 1
            if acc_base > melhor_acc: melhor_acc, melhor_arq = acc_base, arq
            print(f"  GO [{evals:3d}/{max_evals}] acc={acc_base:.4f} best={melhor_acc:.4f} (post-update)")
        step += 1

    return melhor_arq, melhor_acc

## 3. Execução comparativa e análise final

Após as 100 avaliações de busca (sinal = acurácia), a melhor arquitetura de cada SSt
é reavaliada completamente para obter **F1-score macro** e **FLOPs estimados**.

O ranking final usa acurácia como critério primário.
A análise Acc × FLOPs expõe o trade-off performance×custo computacional.


In [ ]:
max_avaliacoes = 100
resultados = {}

otimizadores = {
    "Random Search (RS) ": random_search
    # "Evolutionary (EA)  ": evolutionary_search,
    # "Reinforcement (RL) ": rl_search,
    # "Bayesian (BO)      ": bayesian_search,
    # "DARTS-inspired (GO)": darts_search,
}

print(f"\nDataset : CIFAR-10 (5k treino / 2k val) | Seed: {SEED}")
print(f"Orçamento: {max_avaliacoes} avaliações por SSt | VSt: 10 épocas + EarlyStopping(p=4)")
print(f"SSp: {list(search_space.keys())}")
print("=" * 78)

csv_path = "resultados_micro_nas.csv"

# Cria o arquivo e escreve o cabeçalho ANTES de iniciar as buscas
with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["sst", "acc", "f1", "flops", "tempo_s",
                     "num_layers", "filters", "kernel_size",
                     "activation", "dropout", "dense_units", "optimizer"])

for nome, funcao in otimizadores.items():
    print(f"\n>>> {nome.strip()}")
    random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
    inicio = time.time()
    melhor_arq, _ = funcao(max_avaliacoes)
    tempo_busca = time.time() - inicio

    K.clear_session()
    gc.collect()
    tf.random.set_seed(SEED)

    acc_final, f1_final, flops = avaliar_arquitetura(melhor_arq)
    resultados[nome] = {"acc": acc_final, "f1": f1_final,
                         "flops": flops, "tempo": tempo_busca, "arq": melhor_arq}
    print(f"    => Acc: {acc_final:.4f} | F1: {f1_final:.4f} | "
          f"FLOPs: {flops/1e6:.1f}M | Tempo: {tempo_busca:.0f}s")

    # Salva o resultado desta estratégia iterativamente em modo append ('a')
    with open(csv_path, 'a', newline='') as f:
        writer = csv.writer(f)
        a = melhor_arq
        writer.writerow([nome.strip(), acc_final, f1_final, flops, round(tempo_busca),
                          a['num_layers'], a['filters'], a['kernel_size'],
                          a['activation'], a['dropout'], a['dense_units'], a['optimizer']])


Dataset : CIFAR-10 (5k treino / 2k val) | Seed: 42
Orçamento: 100 avaliações por SSt | VSt: 10 épocas + EarlyStopping(p=4)
SSp: ['num_layers', 'filters', 'kernel_size', 'activation', 'dropout', 'dense_units', 'optimizer']

>>> Random Search (RS)
  RS [  1/100] acc=0.5010 best=0.5010
  RS [  2/100] acc=0.4710 best=0.5010
  RS [  3/100] acc=0.4835 best=0.5010
  RS [  4/100] acc=0.4290 best=0.5010
  RS [  5/100] acc=0.4505 best=0.5010
  RS [  6/100] acc=0.3565 best=0.5010
  RS [  7/100] acc=0.3820 best=0.5010
  RS [  8/100] acc=0.4830 best=0.5010
  RS [  9/100] acc=0.3530 best=0.5010
  RS [ 10/100] acc=0.4510 best=0.5010
  RS [ 11/100] acc=0.4795 best=0.5010
  RS [ 12/100] acc=0.4525 best=0.5010
  RS [ 13/100] acc=0.5085 best=0.5085
  RS [ 14/100] acc=0.3675 best=0.5085
  RS [ 15/100] acc=0.4895 best=0.5085
  RS [ 16/100] acc=0.5315 best=0.5315
  RS [ 17/100] acc=0.5115 best=0.5315
  RS [ 18/100] acc=0.5055 best=0.5315
  RS [ 19/100] acc=0.4635 best=0.5315
  RS [ 20/100] acc=0.4475 best=

In [ ]:
max_avaliacoes = 100

otimizadores = {
    # "Random Search (RS) ": random_search,
    "Evolutionary (EA)  ": evolutionary_search
    # "Reinforcement (RL) ": rl_search,
    # "Bayesian (BO)      ": bayesian_search,
    # "DARTS-inspired (GO)": darts_search,
}

print(f"\nDataset : CIFAR-10 (5k treino / 2k val) | Seed: {SEED}")
print(f"Orçamento: {max_avaliacoes} avaliações por SSt | VSt: 10 épocas + EarlyStopping(p=4)")
print(f"SSp: {list(search_space.keys())}")
print("=" * 78)

csv_path = "resultados_micro_nas.csv"

# Cria o arquivo e escreve o cabeçalho ANTES de iniciar as buscas
with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["sst", "acc", "f1", "flops", "tempo_s",
                     "num_layers", "filters", "kernel_size",
                     "activation", "dropout", "dense_units", "optimizer"])

for nome, funcao in otimizadores.items():
    print(f"\n>>> {nome.strip()}")
    random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
    inicio = time.time()
    melhor_arq, _ = funcao(max_avaliacoes)
    tempo_busca = time.time() - inicio

    K.clear_session()
    gc.collect()
    tf.random.set_seed(SEED)

    acc_final, f1_final, flops = avaliar_arquitetura(melhor_arq)
    resultados[nome] = {"acc": acc_final, "f1": f1_final,
                         "flops": flops, "tempo": tempo_busca, "arq": melhor_arq}
    print(f"    => Acc: {acc_final:.4f} | F1: {f1_final:.4f} | "
          f"FLOPs: {flops/1e6:.1f}M | Tempo: {tempo_busca:.0f}s")

    # Salva o resultado desta estratégia iterativamente em modo append ('a')
    with open(csv_path, 'a', newline='') as f:
        writer = csv.writer(f)
        a = melhor_arq
        writer.writerow([nome.strip(), acc_final, f1_final, flops, round(tempo_busca),
                          a['num_layers'], a['filters'], a['kernel_size'],
                          a['activation'], a['dropout'], a['dense_units'], a['optimizer']])


Dataset : CIFAR-10 (5k treino / 2k val) | Seed: 42
Orçamento: 100 avaliações por SSt | VSt: 10 épocas + EarlyStopping(p=4)
SSp: ['num_layers', 'filters', 'kernel_size', 'activation', 'dropout', 'dense_units', 'optimizer']

>>> Evolutionary (EA)
  EA [  1/100] acc=0.4755 (init)
  EA [  2/100] acc=0.4180 (init)
  EA [  3/100] acc=0.4765 (init)
  EA [  4/100] acc=0.4740 (init)
  EA [  5/100] acc=0.4720 (init)
  EA [  6/100] acc=0.5145 best=0.5145
  EA [  7/100] acc=0.2380 best=0.5145
  EA [  8/100] acc=0.4385 best=0.5145
  EA [  9/100] acc=0.3880 best=0.5145
  EA [ 10/100] acc=0.2560 best=0.5145
  EA [ 11/100] acc=0.5235 best=0.5235
  EA [ 12/100] acc=0.5195 best=0.5235
  EA [ 13/100] acc=0.4975 best=0.5235
  EA [ 14/100] acc=0.4920 best=0.5235
  EA [ 15/100] acc=0.4740 best=0.5235
  EA [ 16/100] acc=0.5105 best=0.5235
  EA [ 17/100] acc=0.1185 best=0.5235
  EA [ 18/100] acc=0.5705 best=0.5705
  EA [ 19/100] acc=0.4575 best=0.5705
  EA [ 20/100] acc=0.4495 best=0.5705
  EA [ 21/100] acc=

In [ ]:
max_avaliacoes = 100

otimizadores = {
    # "Random Search (RS) ": random_search,
    # "Evolutionary (EA)  ": evolutionary_search,
    "Reinforcement (RL) ": rl_search
    # "Bayesian (BO)      ": bayesian_search,
    # "DARTS-inspired (GO)": darts_search,
}

print(f"\nDataset : CIFAR-10 (5k treino / 2k val) | Seed: {SEED}")
print(f"Orçamento: {max_avaliacoes} avaliações por SSt | VSt: 10 épocas + EarlyStopping(p=4)")
print(f"SSp: {list(search_space.keys())}")
print("=" * 78)

csv_path = "resultados_micro_nas.csv"

# Cria o arquivo e escreve o cabeçalho ANTES de iniciar as buscas
with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["sst", "acc", "f1", "flops", "tempo_s",
                     "num_layers", "filters", "kernel_size",
                     "activation", "dropout", "dense_units", "optimizer"])

for nome, funcao in otimizadores.items():
    print(f"\n>>> {nome.strip()}")
    random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
    inicio = time.time()
    melhor_arq, _ = funcao(max_avaliacoes)
    tempo_busca = time.time() - inicio

    K.clear_session()
    gc.collect()
    tf.random.set_seed(SEED)

    acc_final, f1_final, flops = avaliar_arquitetura(melhor_arq)
    resultados[nome] = {"acc": acc_final, "f1": f1_final,
                         "flops": flops, "tempo": tempo_busca, "arq": melhor_arq}
    print(f"    => Acc: {acc_final:.4f} | F1: {f1_final:.4f} | "
          f"FLOPs: {flops/1e6:.1f}M | Tempo: {tempo_busca:.0f}s")

    # Salva o resultado desta estratégia iterativamente em modo append ('a')
    with open(csv_path, 'a', newline='') as f:
        writer = csv.writer(f)
        a = melhor_arq
        writer.writerow([nome.strip(), acc_final, f1_final, flops, round(tempo_busca),
                          a['num_layers'], a['filters'], a['kernel_size'],
                          a['activation'], a['dropout'], a['dense_units'], a['optimizer']])


Dataset : CIFAR-10 (5k treino / 2k val) | Seed: 42
Orçamento: 100 avaliações por SSt | VSt: 10 épocas + EarlyStopping(p=4)
SSp: ['num_layers', 'filters', 'kernel_size', 'activation', 'dropout', 'dense_units', 'optimizer']

>>> Reinforcement (RL)
  RL [  1/100] acc=0.4770 best=0.4770
  RL [  2/100] acc=0.4420 best=0.4770
  RL [  3/100] acc=0.3890 best=0.4770
  RL [  4/100] acc=0.4070 best=0.4770
  RL [  5/100] acc=0.4245 best=0.4770
  RL [  6/100] acc=0.5210 best=0.5210
  RL [  7/100] acc=0.3245 best=0.5210
  RL [  8/100] acc=0.3950 best=0.5210
  RL [  9/100] acc=0.3475 best=0.5210
  RL [ 10/100] acc=0.4160 best=0.5210
  RL [ 11/100] acc=0.4780 best=0.5210
  RL [ 12/100] acc=0.3935 best=0.5210
  RL [ 13/100] acc=0.4740 best=0.5210
  RL [ 14/100] acc=0.3790 best=0.5210
  RL [ 15/100] acc=0.3650 best=0.5210
  RL [ 16/100] acc=0.4670 best=0.5210
  RL [ 17/100] acc=0.5065 best=0.5210
  RL [ 18/100] acc=0.3985 best=0.5210
  RL [ 19/100] acc=0.5100 best=0.5210
  RL [ 20/100] acc=0.4355 best=

In [ ]:
max_avaliacoes = 100

otimizadores = {
    # "Random Search (RS) ": random_search,
    # "Evolutionary (EA)  ": evolutionary_search,
    # "Reinforcement (RL) ": rl_search,
    "Bayesian (BO)      ": bayesian_search
    # "DARTS-inspired (GO)": darts_search,
}

print(f"\nDataset : CIFAR-10 (5k treino / 2k val) | Seed: {SEED}")
print(f"Orçamento: {max_avaliacoes} avaliações por SSt | VSt: 10 épocas + EarlyStopping(p=4)")
print(f"SSp: {list(search_space.keys())}")
print("=" * 78)

csv_path = "resultados_micro_nas.csv"

# Cria o arquivo e escreve o cabeçalho ANTES de iniciar as buscas
with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["sst", "acc", "f1", "flops", "tempo_s",
                     "num_layers", "filters", "kernel_size",
                     "activation", "dropout", "dense_units", "optimizer"])

for nome, funcao in otimizadores.items():
    print(f"\n>>> {nome.strip()}")
    random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
    inicio = time.time()
    melhor_arq, _ = funcao(max_avaliacoes)
    tempo_busca = time.time() - inicio

    K.clear_session()
    gc.collect()
    tf.random.set_seed(SEED)

    acc_final, f1_final, flops = avaliar_arquitetura(melhor_arq)
    resultados[nome] = {"acc": acc_final, "f1": f1_final,
                         "flops": flops, "tempo": tempo_busca, "arq": melhor_arq}
    print(f"    => Acc: {acc_final:.4f} | F1: {f1_final:.4f} | "
          f"FLOPs: {flops/1e6:.1f}M | Tempo: {tempo_busca:.0f}s")

    # Salva o resultado desta estratégia iterativamente em modo append ('a')
    with open(csv_path, 'a', newline='') as f:
        writer = csv.writer(f)
        a = melhor_arq
        writer.writerow([nome.strip(), acc_final, f1_final, flops, round(tempo_busca),
                          a['num_layers'], a['filters'], a['kernel_size'],
                          a['activation'], a['dropout'], a['dense_units'], a['optimizer']])


Dataset : CIFAR-10 (5k treino / 2k val) | Seed: 42
Orçamento: 100 avaliações por SSt | VSt: 10 épocas + EarlyStopping(p=4)
SSp: ['num_layers', 'filters', 'kernel_size', 'activation', 'dropout', 'dense_units', 'optimizer']

>>> Bayesian (BO)
  BO [  1/100] acc=0.5250 (warmup)
  BO [  2/100] acc=0.4850 (warmup)
  BO [  3/100] acc=0.4995 (warmup)
  BO [  4/100] acc=0.4760 (warmup)
  BO [  5/100] acc=0.4500 (warmup)
  BO [  6/100] acc=0.4430 (warmup)
  BO [  7/100] acc=0.4135 (warmup)
  BO [  8/100] acc=0.4585 (warmup)
  BO [  9/100] acc=0.4925 (warmup)
  BO [ 10/100] acc=0.4820 (warmup)
  BO [ 11/100] acc=0.4820 best=0.5250
  BO [ 12/100] acc=0.5505 best=0.5505
  BO [ 13/100] acc=0.5185 best=0.5505
  BO [ 14/100] acc=0.5235 best=0.5505
  BO [ 15/100] acc=0.4825 best=0.5505
  BO [ 16/100] acc=0.4995 best=0.5505
  BO [ 17/100] acc=0.5320 best=0.5505
  BO [ 18/100] acc=0.4480 best=0.5505
  BO [ 19/100] acc=0.4550 best=0.5505
  BO [ 20/100] acc=0.4010 best=0.5505
  BO [ 21/100] acc=0.4235 be

In [ ]:
max_avaliacoes = 100

otimizadores = {
    # "Random Search (RS) ": random_search,
    # "Evolutionary (EA)  ": evolutionary_search,
    # "Reinforcement (RL) ": rl_search,
    # "Bayesian (BO)      ": bayesian_search,
    "DARTS-inspired (GO)": darts_search
}

print(f"\nDataset : CIFAR-10 (5k treino / 2k val) | Seed: {SEED}")
print(f"Orçamento: {max_avaliacoes} avaliações por SSt | VSt: 10 épocas + EarlyStopping(p=4)")
print(f"SSp: {list(search_space.keys())}")
print("=" * 78)

csv_path = "resultados_micro_nas.csv"

# Cria o arquivo e escreve o cabeçalho ANTES de iniciar as buscas
with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["sst", "acc", "f1", "flops", "tempo_s",
                     "num_layers", "filters", "kernel_size",
                     "activation", "dropout", "dense_units", "optimizer"])

for nome, funcao in otimizadores.items():
    print(f"\n>>> {nome.strip()}")
    random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
    inicio = time.time()
    melhor_arq, _ = funcao(max_avaliacoes)
    tempo_busca = time.time() - inicio

    K.clear_session()
    gc.collect()
    tf.random.set_seed(SEED)

    acc_final, f1_final, flops = avaliar_arquitetura(melhor_arq)
    resultados[nome] = {"acc": acc_final, "f1": f1_final,
                         "flops": flops, "tempo": tempo_busca, "arq": melhor_arq}
    print(f"    => Acc: {acc_final:.4f} | F1: {f1_final:.4f} | "
          f"FLOPs: {flops/1e6:.1f}M | Tempo: {tempo_busca:.0f}s")

    # Salva o resultado desta estratégia iterativamente em modo append ('a')
    with open(csv_path, 'a', newline='') as f:
        writer = csv.writer(f)
        a = melhor_arq
        writer.writerow([nome.strip(), acc_final, f1_final, flops, round(tempo_busca),
                          a['num_layers'], a['filters'], a['kernel_size'],
                          a['activation'], a['dropout'], a['dense_units'], a['optimizer']])

In [ ]:
print("\n" + "=" * 78)
print("RANKING FINAL (por acurácia de validação)")
print("=" * 78)
for pos, (nome, r) in enumerate(
        sorted(resultados.items(), key=lambda x: x[1]['acc'], reverse=True), 1):
    print(f"{pos}. {nome} | Acc: {r['acc']:.4f} | F1: {r['f1']:.4f} | "
          f"FLOPs: {r['flops']/1e6:.1f}M | Tempo: {r['tempo']:.0f}s")
    print(f"   Arq: {r['arq']}")

print("\n" + "-" * 78)
print("ANÁLISE: Accuracy vs FLOPs (trade-off performance x custo)")
print("-" * 78)
for nome, r in sorted(resultados.items(), key=lambda x: x[1]['flops']):
    bar = "X" * int(r['acc'] * 40)
    print(f"{nome} | FLOPs: {r['flops']/1e6:5.1f}M | Acc: {r['acc']:.4f} | {bar}")

print(f"\nResultados salvos em: {csv_path}")

try:
    from google.colab import files
    files.download(csv_path)
except ImportError:
    pass